### 1. Importing Necessary Library

In [51]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sklearn
from math import radians, sin, cos, sqrt, asin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor,ExtraTreesRegressor,VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from lightgbm.sklearn import LGBMRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import optuna
from sklearn.model_selection import cross_val_score

### 2. Data Understanding

Defines a function clean_delivery_data(df) that performs initial cleaning:

Strips whitespace from ID columns.

Extracts numeric value from Time_taken(min) (removes any extra text).

Cleans Weatherconditions by removing the prefix "conditions ".

Replaces string 'NaN ' with actual NaN.

Converts columns like Delivery_person_Age, Ratings, etc., to numeric.

Sets latitude/longitude values of 0 to NaN (invalid coordinates).



In [39]:
def clean_delivery_data(df):
    """
    Basic cleaning operations for the delivery dataset

    Parameters:
    df: DataFrame to clean

    Returns:
    Cleaned DataFrame
    """
    # Make a copy to avoid modifying original
    df = df.copy()

    # Clean ID columns
    if 'ID' in df.columns:
        df['ID'] = df['ID'].str.strip()

    if 'Delivery_person_ID' in df.columns:
        df['Delivery_person_ID'] = df['Delivery_person_ID'].str.strip()

    # Clean Time_taken column - extract numeric value
    if 'Time_taken(min)' in df.columns:
        df['Time_taken(min)'] = df['Time_taken(min)'].astype(str).str.extract(r'(\d+)').astype(float)

    # Clean Weatherconditions
    if 'Weatherconditions' in df.columns:
        df['Weatherconditions'] = df['Weatherconditions'].astype(str).str.replace('conditions ', '', regex=False)

    # Clean Road_traffic_density
    if 'Road_traffic_density' in df.columns:
        df['Road_traffic_density'] = df['Road_traffic_density'].astype(str).str.strip()

    # Replace 'NaN ' strings with actual NaN
    df = df.replace('NaN ', np.nan)
    df = df.replace('NaN', np.nan)

    # Convert numeric columns
    numeric_cols = ['Delivery_person_Age', 'Delivery_person_Ratings',
                    'Vehicle_condition', 'multiple_deliveries']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fix latitude/longitude issues (mark zeros as invalid)
    location_cols = ['Restaurant_latitude', 'Restaurant_longitude',
                     'Delivery_location_latitude', 'Delivery_location_longitude']
    for col in location_cols:
        if col in df.columns:
            df.loc[df[col] == 0, col] = np.nan

    return df


Defines extract_date_features(df) which:

Converts Order_Date to datetime.

Creates new columns: Year, Month, Day, DayOfWeek, IsWeekend, and DayName (e.g., Monday).
These features help capture seasonal or weekly patterns in delivery times.

In [40]:

def extract_date_features(df):
    """
    Extract features from Order_Date column

    Parameters:
    df: DataFrame with Order_Date column

    Returns:
    DataFrame with date features
    """
    df = df.copy()

    if 'Order_Date' in df.columns:
        # Convert to datetime
        df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True, errors='coerce')

        # Extract date components
        df['Year'] = df['Order_Date'].dt.year
        df['Month'] = df['Order_Date'].dt.month
        df['Day'] = df['Order_Date'].dt.day
        df['DayOfWeek'] = df['Order_Date'].dt.dayofweek
        df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

        # Day name for reference
        day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
                   4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
        df['DayName'] = df['DayOfWeek'].map(day_map)

    return df


Defines extract_time_features(df) to process the time columns Time_Orderd and Time_Order_picked:

Parses time strings into time objects.

Extracts hour and categorises into Morning, Afternoon, Evening, Night.

Computes Time_to_assign = (pickup time – order time) in minutes, handling cases where pickup occurs after midnight.

In [41]:
def extract_time_features(df):
    """
    Extract features from time columns

    Parameters:
    df: DataFrame with Time_Orderd and Time_Order_picked columns

    Returns:
    DataFrame with time features
    """
    df = df.copy()

    def parse_time(time_str):
        """Helper function to parse time strings"""
        try:
            if pd.isna(time_str) or time_str == 'NaN ':
                return np.nan
            return pd.to_datetime(time_str, format='%H:%M:%S').time()
        except:
            return np.nan

    def time_diff(order_time, picked_time):
        """Calculate time difference in minutes"""
        if pd.isna(order_time) or pd.isna(picked_time):
            return np.nan
        try:
            order = datetime.combine(datetime.today(), order_time)
            picked = datetime.combine(datetime.today(), picked_time)
            diff = (picked - order).total_seconds() / 60
            if diff < 0:  # Handle cases where pickup is on next day
                diff += 24 * 60
            return diff
        except:
            return np.nan

    def get_hour(time_obj):
        """Extract hour from time object"""
        if pd.isna(time_obj):
            return np.nan
        return time_obj.hour

    def get_time_category(hour):
        """Categorize time of day"""
        if pd.isna(hour):
            return np.nan
        if 5 <= hour < 12:
            return 'Morning'
        elif 12 <= hour < 17:
            return 'Afternoon'
        elif 17 <= hour < 21:
            return 'Evening'
        else:
            return 'Night'

    if 'Time_Orderd' in df.columns:
        df['Time_Orderd'] = df['Time_Orderd'].apply(parse_time)
        df['Order_Hour'] = df['Time_Orderd'].apply(get_hour)
        df['Order_Time_Category'] = df['Order_Hour'].apply(get_time_category)

    if 'Time_Order_picked' in df.columns:
        df['Time_Order_picked'] = df['Time_Order_picked'].apply(parse_time)
        df['Pickup_Hour'] = df['Time_Order_picked'].apply(get_hour)
        df['Pickup_Time_Category'] = df['Pickup_Hour'].apply(get_time_category)

    # Calculate time to assign if both times are available
    if 'Time_Orderd' in df.columns and 'Time_Order_picked' in df.columns:
        df['Time_to_assign'] = df.apply(
            lambda x: time_diff(x['Time_Orderd'], x['Time_Order_picked']), axis=1
        )

    return df


Defines calculate_distance(df) which uses the haversine formula to compute the great‑circle distance (in km) between the restaurant’s coordinates and the delivery location. The result is stored in a new column Distance_km. Missing coordinates are handled gracefully.



In [42]:

def calculate_distance(df):
    """
    Calculate haversine distance between restaurant and delivery location

    Parameters:
    df: DataFrame with latitude/longitude columns

    Returns:
    DataFrame with distance column
    """
    df = df.copy()

    def haversine(lat1, lon1, lat2, lon2):
        """Calculate great-circle distance between two points"""
        if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
            return np.nan

        R = 6371  # Earth's radius in km

        try:
            lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
            dlat = lat2 - lat1
            dlon = lon2 - lon1

            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
            c = 2 * np.arcsin(np.sqrt(a))

            return R * c
        except:
            return np.nan

    required_cols = ['Restaurant_latitude', 'Restaurant_longitude',
                     'Delivery_location_latitude', 'Delivery_location_longitude']

    if all(col in df.columns for col in required_cols):
        df['Distance_km'] = df.apply(
            lambda x: haversine(
                x['Restaurant_latitude'], x['Restaurant_longitude'],
                x['Delivery_location_latitude'], x['Delivery_location_longitude']
            ), axis=1
        )

    return df


Defines extract_id_features(df) to parse Delivery_person_ID:

Extracts the city code (e.g., "BANGLORE" → "BANGLORE").

Extracts the person number (digits after RES).
These can be useful for capturing driver‑specific effects.

In [43]:

def extract_id_features(df):
    """
    Extract features from Delivery_person_ID

    Parameters:
    df: DataFrame with Delivery_person_ID column

    Returns:
    DataFrame with extracted ID features
    """
    df = df.copy()

    if 'Delivery_person_ID' in df.columns:
        # Extract city code
        df['Delivery_city'] = df['Delivery_person_ID'].astype(str).str.extract(r'([A-Z]+)')

        # Extract person number
        df['Delivery_person_num'] = df['Delivery_person_ID'].astype(str).str.extract(r'RES(\d+)')
        df['Delivery_person_num'] = pd.to_numeric(df['Delivery_person_num'], errors='coerce')

    return df


Defines create_interaction_features(df) to generate additional features:

Age groups, rating groups, distance groups (categorical bins).

- Interaction features like Age_Rating_Interaction (product of age and rating) and Distance_Vehicle_Interaction (distance × vehicle condition).

- ⚠️ Note: The commented‑out speed feature was removed because it would leak the target variable (Time_taken(min)).



In [44]:
def create_interaction_features(df):
    """
    Create interaction features from existing columns

    Parameters:
    df: DataFrame with base features

    Returns:
    DataFrame with interaction features
    """
    df = df.copy()

    # Speed feature (if both distance and time are available) - REMOVED due to target leakage
    # if 'Distance_km' in df.columns and 'Time_taken(min)' in df.columns:
    #     df['Speed_kmph'] = df['Distance_km'] / (df['Time_taken(min)'] / 60)
    #     # Handle infinite values
    #     df['Speed_kmph'] = df['Speed_kmph'].replace([np.inf, -np.inf], np.nan)

    # Age groups
    if 'Delivery_person_Age' in df.columns:
        df['Age_Group'] = pd.cut(
            df['Delivery_person_Age'],
            bins=[0, 25, 35, 45, 100],
            labels=['Young', 'Middle', 'Senior', 'Very Senior']
        )

    # Rating groups
    if 'Delivery_person_Ratings' in df.columns:
        df['Rating_Group'] = pd.cut(
            df['Delivery_person_Ratings'],
            bins=[0, 4.0, 4.5, 4.8, 5.0],
            labels=['Low', 'Medium', 'High', 'Very High']
        )

    # Distance groups
    if 'Distance_km' in df.columns:
        df['Distance_Group'] = pd.cut(
            df['Distance_km'],
            bins=[0, 5, 10, 15, 20, 50],
            labels=['Very Short', 'Short', 'Medium', 'Long', 'Very Long']
        )

    # Interaction features
    if 'Delivery_person_Age' in df.columns and 'Delivery_person_Ratings' in df.columns:
        df['Age_Rating_Interaction'] = df['Delivery_person_Age'] * df['Delivery_person_Ratings']

    if 'Distance_km' in df.columns and 'Vehicle_condition' in df.columns:
        df['Distance_Vehicle_Interaction'] = df['Distance_km'] * (df['Vehicle_condition'].fillna(1) + 1)

    return df

Defines encode_categorical_features() to label‑encode all categorical columns (e.g., weather, traffic, vehicle type, etc.).

For training data (fit_encoders=True), it fits a new LabelEncoder for each column and stores them in a dictionary.

For test data, it uses the previously fitted encoders. Any unseen category is mapped to 'Unknown' (which is added to the encoder’s classes) to avoid errors.

In [45]:
def encode_categorical_features(df, fit_encoders=True, encoders=None):
    """
    Encode categorical features using LabelEncoder

    Parameters:
    df: DataFrame with categorical features
    fit_encoders: Whether to fit new encoders (True for train, False for test)
    encoders: Dictionary of fitted encoders (required when fit_encoders=False)

    Returns:
    DataFrame with encoded features and encoders dictionary
    """
    df = df.copy()

    categorical_cols = [
        'Weatherconditions', 'Road_traffic_density', 'Type_of_order',
        'Type_of_vehicle', 'Festival', 'City', 'Delivery_city',
        'Order_Time_Category', 'Pickup_Time_Category', 'Age_Group',
        'Rating_Group', 'Distance_Group', 'DayName'
    ]

    # Filter to only columns that exist in dataframe
    cols_to_encode = [col for col in categorical_cols if col in df.columns]

    if fit_encoders:
        encoders = {}
        for col in cols_to_encode:
            # Convert to object type before filling NaN to allow new string values
            # This handles cases where columns might be Categorical from pd.cut
            df[col] = df[col].astype(object).fillna('Unknown')
            le = LabelEncoder()
            # Fit and transform on the string representation of the column
            df[col + '_Encoded'] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        return df, encoders
    else:
        if encoders is None:
            raise ValueError("encoders dictionary must be provided when fit_encoders=False")

        for col in cols_to_encode:
            if col in encoders:
                # Convert to object type before filling NaN
                df[col] = df[col].astype(object).fillna('Unknown')

                # Get existing classes from the fitted encoder
                le = encoders[col]
                known_classes = set(le.classes_)

                # Ensure 'Unknown' is in the encoder's classes before transformation
                if 'Unknown' not in known_classes:
                    # Append 'Unknown' to the encoder's classes.
                    # This is a direct modification of LabelEncoder's internal state.
                    le.classes_ = np.append(le.classes_, 'Unknown')
                    known_classes.add('Unknown') # Update known_classes set for the next step

                # Map values that are not in the (potentially updated) known_classes to 'Unknown'
                # Ensure the column is string type for consistent comparison and application
                df[col + '_Processed'] = df[col].astype(str).apply(
                    lambda x: x if x in known_classes else 'Unknown'
                )
                # Now transform the processed column
                df[col + '_Encoded'] = le.transform(df[col + '_Processed'])
                df = df.drop(columns=[col + '_Processed']) # Drop the temporary column

        return df, encoders

This is the master preprocessing pipeline. It reads the CSV and sequentially applies all the previously defined functions:

- clean_delivery_data

- extract_date_features

- extract_time_features

- calculate_distance

- extract_id_features

- create_interaction_features

encode_categorical_features (with the appropriate fit_encoders flag)

If is_train=True, it returns the processed DataFrame and the fitted encoders. If is_train=False, it returns only the DataFrame (using the supplied encoders). Progress messages are printed.

In [46]:

def process_delivery_data(file_path, is_train=True, encoders=None):
    """
    Main function to process delivery data

    Parameters:
    file_path: Path to the CSV file
    is_train: Whether this is training data (True) or test data (False)
    encoders: Dictionary of fitted encoders (required for test data)

    Returns:
    Processed DataFrame and encoders (for train) or just DataFrame (for test)
    """
    print(f"Loading data from {file_path}...")
    df = pd.read_csv(file_path)
    print(f"Initial shape: {df.shape}")

    # Step 1: Basic cleaning
    print("Step 1: Performing basic cleaning...")
    df = clean_delivery_data(df)

    # Step 2: Extract date features
    print("Step 2: Extracting date features...")
    df = extract_date_features(df)

    # Step 3: Extract time features
    print("Step 3: Extracting time features...")
    df = extract_time_features(df)

    # Step 4: Calculate distance
    print("Step 4: Calculating distance...")
    df = calculate_distance(df)

    # Step 5: Extract ID features
    print("Step 5: Extracting ID features...")
    df = extract_id_features(df)

    # Step 6: Create interaction features
    print("Step 6: Creating interaction features...")
    df = create_interaction_features(df)

    # Step 7: Encode categorical features
    print("Step 7: Encoding categorical features...")
    if is_train:
        df, encoders = encode_categorical_features(df, fit_encoders=True)
    else:
        df, encoders = encode_categorical_features(df, fit_encoders=False, encoders=encoders)

    print(f"Final shape: {df.shape}")
    print(f"Total features: {len(df.columns)}")

    if is_train:
        return df, encoders
    else:
        return df


Defines get_feature_columns() to return a list of numeric columns that should be used as features. It excludes:

- Identifier columns (ID, Delivery_person_ID)

- Original date/time columns (Order_Date, Time_Orderd, Time_Order_picked, DayName)

- The target variable Time_taken(min) if exclude_target=True

In [47]:


def get_feature_columns(df, exclude_target=True):
    """
    Get list of feature columns (excluding target and ID columns)

    Parameters:
    df: Processed DataFrame
    exclude_target: Whether to exclude target column

    Returns:
    List of feature column names
    """
    # Columns to exclude from features
    exclude_cols = ['ID', 'Delivery_person_ID', 'Order_Date', 'Time_Orderd',
                    'Time_Order_picked', 'DayName']

    if exclude_target and 'Time_taken(min)' in df.columns:
        exclude_cols.append('Time_taken(min)')

    # Get all numeric columns that aren't in exclude list
    feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col not in exclude_cols]

    return feature_cols



- Loads and processes the training data using process_delivery_data.

- Defines the feature matrix X and target vector y.

- Uses SimpleImputer with mean strategy to fill missing values in X (the imputer is fitted on the whole dataset – note: this is done before splitting, which is acceptable here because the split is sequential and we are not leaking future information into the imputation).

- Splits the data into training (first 80%) and testing (last 20%). This is a sequential split (not random), which is appropriate for time‑ordered data.

In [48]:
from sklearn.impute import SimpleImputer

train_df, encoders = process_delivery_data('train.csv', is_train=True)

feature_cols = get_feature_columns(train_df)

X = train_df[feature_cols]
y = train_df['Time_taken(min)']

# Impute missing values in X before splitting
imputer = SimpleImputer(strategy='mean')
X_imputed_array = imputer.fit_transform(X)
# Get the feature names that the imputer actually processed and outputted
imputed_feature_names = imputer.get_feature_names_out(input_features=X.columns)
X_imputed = pd.DataFrame(X_imputed_array, columns=imputed_feature_names, index=X.index)

# Define split index
split_idx = int(len(X_imputed) * 0.8)

# Proper train-test split
X_train = X_imputed.iloc[:split_idx].copy()
X_test = X_imputed.iloc[split_idx:].copy()

y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

Loading data from train.csv...
Initial shape: (45593, 20)
Step 1: Performing basic cleaning...
Step 2: Extracting date features...
Step 3: Extracting time features...
Step 4: Calculating distance...
Step 5: Extracting ID features...
Step 6: Creating interaction features...
Step 7: Encoding categorical features...
Final shape: (45593, 52)
Total features: 52


E:\Miniconda3\envs\ds\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['Time_to_assign']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


### 5. ML Model Building

Defines two helper functions:

- model_eval(model): Makes predictions on the training and test sets, then prints and returns a list of eight metrics: R² train/test, RSS train/test, MSE train/test, RMSE train/test.

- store_model_metrics(model_name, metric_list): Stores these metrics in a global dictionary all_model_metrics for later comparison.

In [49]:
def model_eval(model):

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metric = []
    r2_train_lr = r2_score(y_train, y_pred_train)
    print("R2 score for train:", r2_train_lr)
    metric.append(r2_train_lr)

    r2_test_lr = r2_score(y_test, y_pred_test)
    print("R2 score for test:", r2_test_lr)
    metric.append(r2_test_lr)
    print("--"*20)

    rss1_lr = np.sum(np.square(y_train - y_pred_train))
    print("RSS for train:", rss1_lr)
    metric.append(rss1_lr)

    rss2_lr = np.sum(np.square(y_test - y_pred_test))
    print("RSS for test:", rss2_lr)
    metric.append(rss2_lr)
    print("--"*20)

    mse_train_lr = mean_squared_error(y_train, y_pred_train)
    print("MSE for train:", mse_train_lr)
    metric.append(mse_train_lr)

    mse_test_lr = mean_squared_error(y_test, y_pred_test)
    print("MSE for test:", mse_test_lr)
    metric.append(mse_test_lr)
    print("--"*20)

    rmse_train_lr = mse_train_lr**0.5
    print("RMSE for train:", rmse_train_lr)
    metric.append(rmse_train_lr)

    rmse_test_lr = mse_test_lr**0.5
    print("RMSE for test:", rmse_test_lr)
    metric.append(rmse_test_lr)
    print("--"*20)

    return metric, y_pred_train, y_pred_test

all_model_metrics = {}

def store_model_metrics(model_name, metric_list):
    all_model_metrics[model_name] = {
        'R2_Train': metric_list[0],
        'R2_Test': metric_list[1],
        'RSS_Train': metric_list[2],
        'RSS_Test': metric_list[3],
        'MSE_Train': metric_list[4],
        'MSE_Test': metric_list[5],
        'RMSE_Train': metric_list[6],
        'RMSE_Test': metric_list[7]
    }
    return all_model_metrics[model_name]

2. XGBRegressor

Performs hyperparameter optimisation for XGBoost using Optuna:

- The objective function defines the hyperparameter search space (integers, floats) and returns the mean negative RMSE from 5‑fold cross‑validation (Optuna maximises this, i.e., minimises RMSE).

- The study runs for 50 trials.

- After optimisation, the best set of parameters is printed.

In [52]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "objective": "reg:squarederror"
    }

    model = xgb.XGBRegressor(**params)

    # 5-fold cross validation
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    return np.mean(score)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best Parameters:", study.best_params)

[I 2026-02-25 17:31:54,085] A new study created in memory with name: no-name-f90c937c-83d7-47a8-8b29-20d7faccd15a
[I 2026-02-25 17:31:59,564] Trial 0 finished with value: -4.2836242598816305 and parameters: {'n_estimators': 278, 'max_depth': 7, 'learning_rate': 0.27023694985940755, 'subsample': 0.7917903652035972, 'colsample_bytree': 0.9240473698562648, 'gamma': 3.2767738449878814, 'reg_alpha': 1.374638177729011, 'reg_lambda': 4.327031874877608}. Best is trial 0 with value: -4.2836242598816305.
[I 2026-02-25 17:32:10,060] Trial 1 finished with value: -4.0892614648207175 and parameters: {'n_estimators': 812, 'max_depth': 8, 'learning_rate': 0.1258872559921982, 'subsample': 0.8711988859537936, 'colsample_bytree': 0.6680350618965012, 'gamma': 4.231979839571931, 'reg_alpha': 3.6388365160271317, 'reg_lambda': 3.087751187600223}. Best is trial 1 with value: -4.0892614648207175.
[I 2026-02-25 17:32:24,682] Trial 2 finished with value: -4.2619292515401055 and parameters: {'n_estimators': 796, 

Best Parameters: {'n_estimators': 521, 'max_depth': 10, 'learning_rate': 0.014563056590600915, 'subsample': 0.8478747824317217, 'colsample_bytree': 0.8588430515567245, 'gamma': 2.6708720097496923, 'reg_alpha': 0.7678671001832196, 'reg_lambda': 4.693006617637542}


- Instantiates an XGBoost regressor with the best parameters found by Optuna.

- Trains the model on the full training set (X_train, y_train).

- Evaluates it using model_eval() and stores the metrics under the name 'XGBoost_Optuna'.

Finally, prints the list of metrics.

In [54]:
# Create final model using best parameters
best_xgb = xgb.XGBRegressor(
    **study.best_params,
    random_state=42,
    objective="reg:squarederror"
)

# Train
best_xgb.fit(X_train, y_train)

# Use your existing evaluation function
metric_xgb_optuna, y_pred_train, y_pred_test = model_eval(best_xgb)

# Store metrics
store_model_metrics('XGBoost_Optuna', metric_xgb_optuna)

print(metric_xgb_optuna)

R2 score for train: 0.9043506980274366
R2 score for test: 0.83135991785022
----------------------------------------
RSS for train: 306104.31920249097
RSS for test: 137320.9829385675
----------------------------------------
MSE for train: 8.392397850591955
MSE for test: 15.058776503845541
----------------------------------------
RMSE for train: 2.896963557001012
RMSE for test: 3.880563941471077
----------------------------------------
[0.9043506980274366, 0.83135991785022, 306104.31920249097, 137320.9829385675, 8.392397850591955, 15.058776503845541, 2.896963557001012, 3.880563941471077]
